# Cell Setup

Python 3.10+. Jalankan berurutan; instalasi opsional, tanpa FastText atau unduhan data.
Resource NLTK diperiksa/diunduh hanya di setup; kegagalan tidak memakai tokenizer pengganti.

In [ ]:
# Opsional: jalankan sekali, lalu restart kernel.
# %pip install "torch>=2.2,<3" "numpy>=1.26,<3" "pandas>=2.1,<3" "scikit-learn>=1.4,<2" "nltk>=3.9.1,<4" "ipykernel>=6.29,<8"
import gc
import hashlib
import importlib.metadata
import json
import platform
import random
import re
import string
import sys
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

if sys.version_info < (3, 10):
    raise RuntimeError("Gunakan Python 3.10 atau lebih baru.")
try:
    nltk.data.find("tokenizers/punkt_tab/english/")
except LookupError:
    if not nltk.download("punkt_tab", quiet=True, raise_on_error=True):
        raise RuntimeError("Instal resource NLTK punkt_tab, kemudian ulangi Setup.")
    nltk.data.find("tokenizers/punkt_tab/english/")

## Cell 1 — Konfigurasi dan batas reproduksi

Target: **MT-Text-CNN scratch Table 4**, bukan model final FastText (F1 0.611).
Paper §4.1.1 memakai vektor acak dan default Kim (2014): kernel `[3,4,5]`,
100 filter/kernel, dropout 0.5, batch 50, Adadelta; max-norm 3 diwarisi dari Kim.
Embedding 300 trainable dan empat head lima kelas berbagi embedding/konvolusi.

**Pilihan implementasi:** 10 epoch tetap tanpa early stopping, panjang 100,
minfreq 2, seed 42, embedding uniform(-0.25,0.25), Adadelta lr=1/rho=.9/eps=1e-6.
Nilai ini tidak semuanya ditentukan paper; tidak mengklaim reproduksi 100% identik.
Loss default **tanpa bobot**: weighted loss baru ditambahkan di ablation §4.1.2
(setelah FastText: .549 menjadi .593). Opsi bobot bukan baseline Table 4;
formulanya tidak diberikan paper, pilihan di sini `n/(5*n_c)`, train-only, tanpa clip.
Detector resmi memakai konfigurasi **final tuned** `[1,2,3,4,5]`/200 filter,
bukan bukti konfigurasi scratch. Urutan head resmi dipertahankan.

In [ ]:
PROJECT_ROOT = None  # Opsional: lokasi penyimpanan hasil; default direktori aktif.
N_SPLITS = 10
QUICK = False
RUN_FOLDS = (1,) if QUICK else tuple(range(1, N_SPLITS + 1))
EPOCHS = 10
BATCH_SIZE = 50
MAX_LEN = 100
MIN_FREQ = 2
SEED = 42
EMBED_DIM = 300
KERNEL_SIZES = (3, 4, 5)
KERNEL_NUM = 100
DROPOUT = 0.5
MAX_NORM = 3.0
LEARNING_RATE = 1.0
ADADELTA_RHO = 0.9
ADADELTA_EPS = 1e-6
USE_CLASS_WEIGHTS = False  # True = varian eksplisit, bukan baseline Table 4.
SAVE_CHECKPOINTS = False  # Simpan epoch terakhir per fold, bukan terbaik menurut test.

SOURCES = ("issue", "commit_message", "code_comment", "pull_request")
TASK_ID = {source: i for i, source in enumerate(SOURCES)}
LABELS = ("non-SATD", "code/design", "requirement", "documentation", "test")
DEBT_CLASSES = (1, 2, 3, 4)
FILES = {'issue': 'https://raw.githubusercontent.com/NuriInformatics/retrain-multi-domain/refs/heads/feat/satd-notebook-reproduction/data/satd-dataset-issues.csv', 'commit_message': 'https://raw.githubusercontent.com/NuriInformatics/retrain-multi-domain/refs/heads/feat/satd-notebook-reproduction/data/satd-dataset-commit_messages.csv', 'code_comment': 'https://github.com/NuriInformatics/retrain-multi-domain/raw/refs/heads/feat/satd-notebook-reproduction/data/satd-dataset-code_comments.csv', 'pull_request': 'https://raw.githubusercontent.com/NuriInformatics/retrain-multi-domain/refs/heads/feat/satd-notebook-reproduction/data/satd-dataset-pull_requests.csv'}
EXPECTED_COUNTS = {
    "code_comment": [58676, 2703, 757, 54, 85],
    "commit_message": [4295, 522, 27, 98, 58],
    "pull_request": [4301, 510, 20, 101, 68],
    "issue": [20089, 2169, 97, 487, 338],
}
PAPER_DOI = "10.1007/s10664-023-10297-9"
SOURCE_COMMIT = "df412d88c12c42164dc7574389070f7ef2cc1d12"
SOURCE_URL = f"https://github.com/yikun-li/satd-different-sources-data/blob/{SOURCE_COMMIT}/SATD%20Detector/satd_detector.py"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

assert N_SPLITS == 10 and EPOCHS > 0 and BATCH_SIZE > 0
assert MAX_LEN >= max(KERNEL_SIZES)
assert RUN_FOLDS and len(set(RUN_FOLDS)) == len(RUN_FOLDS)
assert set(RUN_FOLDS) <= set(range(1, N_SPLITS + 1))
set_seed(SEED)
CV_STATUS = "full_10_fold" if len(RUN_FOLDS) == N_SPLITS else "partial_10_fold"
print(f"Device={DEVICE} | {CV_STATUS} | fold={RUN_FOLDS}")

## Cell 2 — Data dari URL dan label eksplisit

Empat CSV dibaca dari GitHub. Tidak membuang baris rusak atau melakukan deduplikasi.
Teks kosong tetap `""`; `raw_text`, `project`, dan `row_id` indeks asli dipertahankan.
Label di luar pemetaan dihentikan sebagai error.


In [ ]:
from io import BytesIO
from urllib.request import urlopen

ROOT = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT is not None else Path.cwd().resolve()
data_bytes = {}
LABEL_MAP = {
    "non_debt": 0, "defect_debt": 0, "build_debt": 0, "architecture_debt": 0,
    "code_debt": 1, "design_debt": 1, "code/design_debt": 1,
    "requirement_debt": 2, "documentation_debt": 3, "test_debt": 4,
}
frames = {}
for source, url in FILES.items():
    with urlopen(url) as response:
        data_bytes[source] = response.read()
    frame = pd.read_csv(
        BytesIO(data_bytes[source]), usecols=["text", "classification", "project"],
        dtype=str, encoding="utf-8", keep_default_na=False, on_bad_lines="error",
    )
    unknown = set(frame["classification"]) - set(LABEL_MAP)
    if unknown:
        raise ValueError(f"{source}: label tidak dikenal {sorted(unknown)}")
    frame.insert(0, "row_id", frame.index.to_numpy(copy=True))
    frame = frame.rename(columns={"text": "raw_text"})
    frame["y"] = frame["classification"].map(LABEL_MAP).astype("int64")
    counts = np.bincount(frame["y"], minlength=5).tolist()
    assert counts == EXPECTED_COUNTS[source], (source, counts)
    frames[source] = frame
    print(source, len(frame), counts)
print("PERINGATAN: row-level CV mempertahankan duplikat; teks/proyek serupa dapat melintasi train/test dan sumber.")
print("Duplikat teks mentah ekstra (semua sumber):", pd.concat([f["raw_text"] for f in frames.values()]).duplicated().sum())

## Cell 3 — Pembersihan detector-compatible

Kompatibel dengan `satd_detector.py`: hapus penanda komentar, lowercase, `sent_tokenize`,
`TweetTokenizer`, filter simbol/angka; pertahankan `!`, `?`, `non-nls`.
Tanpa stemming/stopword removal; tanpa fallback tokenizer. Padding dilakukan kemudian.
Pembersihan dataset paper §3.4 berbeda: filtering bot, blok kode, dan teks non-Inggris
pada CSV terkurasi upstream tidak divalidasi ulang secara independen di notebook ini.

In [ ]:
tokenizer = nltk.TweetTokenizer()
punctuation = string.punctuation.replace("!", "").replace("?", "")

def clean_text(text):
    text = re.sub(r"(//)|(/\*)|(\*/)", "", text).lower()
    sentences = [tokenizer.tokenize(sentence) for sentence in nltk.sent_tokenize(text)]
    processed = []
    for sentence in sentences:
        kept = []
        for token in sentence:
            if token in ("non-nls", "<pad>"):
                kept.append(token)
            elif token == ",":
                kept.append(".")
            elif any(symbol in token for symbol in (" ", ".", "#", "_", "/", ">", "<", "@")):
                continue
            elif any(char.isdigit() for char in token):
                continue
            else:
                kept.append(token)
        if kept and kept[-1] != ".":
            kept.append(".")
        processed.extend(kept)
    return [token for token in processed if token and token not in punctuation
            and not any(symbol in token for symbol in (":", "=", ")", "("))]

for source in SOURCES:
    frames[source]["tokens"] = frames[source]["raw_text"].map(clean_text)
    print(source, "selesai; teks tanpa token:", frames[source]["tokens"].map(len).eq(0).sum())

## Cell 4 — Partisi stratified 10-fold

Split independen per sumber, shuffle seed 42. QUICK hanya memilih fold pertama
**dari sepuluh partisi yang sama**, bukan mengubah jumlah split.
Ini row-level CV; bukan project-disjoint atau duplicate-disjoint CV.

In [ ]:
splits, fold_tables = {}, []
for source in SOURCES:
    frame = frames[source]
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    splits[source] = list(splitter.split(np.zeros(len(frame)), frame["y"]))
    assignment = np.zeros(len(frame), dtype=np.int64)
    for fold, (train_idx, test_idx) in enumerate(splits[source], start=1):
        assert not np.intersect1d(train_idx, test_idx).size
        assert not assignment[test_idx].any()
        assignment[test_idx] = fold
    assert (assignment > 0).all()
    fold_tables.append(pd.DataFrame({"source": source, "row_id": frame["row_id"], "fold": assignment}))
fold_assignments = pd.concat(fold_tables, ignore_index=True)
assert not fold_assignments.duplicated(["source", "row_id"]).any()
print(f"10 partisi/sumber; dipilih {len(RUN_FOLDS)}/10 fold.")

## Cell 5 — Vocabulary baru per fold

Vocabulary hanya dari partisi **train keempat sumber** pada fold tersebut.
PAD=0, UNK=1, minfreq=2. Token test tidak menambah vocabulary.

In [ ]:
def build_vocab(train_frames):
    counts = Counter(
        token
        for frame in train_frames.values()
        for tokens in frame["tokens"]
        for token in tokens
    )
    vocab = {"<pad>": 0, "<unk>": 1}
    for token in sorted(counts):
        if counts[token] >= MIN_FREQ and token not in vocab:
            vocab[token] = len(vocab)
    return vocab

**Mengubah token menjadi angka.** Token yang tidak ada di vocabulary memakai ID `<unk>`; urutan dipotong atau ditambah padding hingga panjang yang ditentukan.

In [ ]:
def encode(tokens, vocab):
    ids = [vocab.get(token, 1) for token in tokens[:MAX_LEN]]
    length = len(ids)
    return ids + [0] * (MAX_LEN - length), length

## Cell 6 — Dataset dan loss per task

Kelas 0 tetap dilatih. Loss baseline unweighted. Bila opsi bobot diaktifkan,
hitung `n/(5*n_c)` hanya dari train task/fold; tanpa clipping.
Loader shuffle memakai generator terpisah yang berlanjut saat iterator dibuat ulang.

In [ ]:
class SATDDataset(Dataset):
    def __init__(self, frame, vocab):
        encoded = [encode(tokens, vocab) for tokens in frame["tokens"]]
        self.x = torch.tensor([item[0] for item in encoded], dtype=torch.long)
        self.lengths = torch.tensor([item[1] for item in encoded], dtype=torch.long)
        self.y = torch.tensor(frame["y"].to_numpy(), dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.lengths[index], self.y[index]

**Loader dan loss.** Loader membagi data menjadi batch. Bobot kelas hanya dipakai jika `USE_CLASS_WEIGHTS = True`.

In [ ]:
def make_loader(frame, vocab, shuffle, seed):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        SATDDataset(frame, vocab),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
        drop_last=False,
    )

def class_weights(y_train):
    counts = np.bincount(np.asarray(y_train), minlength=5)
    assert (counts > 0).all(), "Setiap kelas harus ada pada train."
    return torch.tensor(len(y_train) / (5 * counts), dtype=torch.float32, device=DEVICE)

def make_criterion(y_train):
    weights = class_weights(y_train) if USE_CLASS_WEIGHTS else None
    return nn.CrossEntropyLoss(weight=weights)

## Cell 7 — MT-Text-CNN scratch

Conv2d melintasi seluruh lebar embedding; tiga kernel dan empat head task-specific.
Inisialisasi embedding uniform(-.25,.25) adalah pilihan implementasi; PAD nol.
Pooling hanya menerima window dalam `max(5, panjang_asli)` agar padding 100
tidak menambah kandidat bias konvolusi. Masking ini adalah pengamanan implementasi.

In [ ]:
class MTTextCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        nn.init.uniform_(self.embed.weight, -0.25, 0.25)
        with torch.no_grad():
            self.embed.weight[0].zero_()
        self.convs = nn.ModuleList([nn.Conv2d(1, KERNEL_NUM, (size, EMBED_DIM)) for size in KERNEL_SIZES])
        self.dropout = nn.Dropout(DROPOUT)
        self.heads = nn.ModuleList([nn.Linear(len(KERNEL_SIZES) * KERNEL_NUM, 5) for _ in SOURCES])

    def forward(self, x, lengths, task_id):
        embedded = self.embed(x).unsqueeze(1)
        effective_lengths = lengths.clamp(min=max(KERNEL_SIZES), max=x.shape[1])
        pooled = []
        for size, conv in zip(KERNEL_SIZES, self.convs):
            features = F.relu(conv(embedded)).squeeze(3)
            positions = torch.arange(features.shape[2], device=x.device)[None, :]
            valid = positions < (effective_lengths - size + 1)[:, None]
            features = features.masked_fill(~valid[:, None, :], float("-inf"))
            pooled.append(features.amax(dim=2))
        return self.heads[task_id](self.dropout(torch.cat(pooled, dim=1)))

    @torch.no_grad()
    def constrain_heads(self):
        # Kim: batas L2 tiap vektor bobot output, bukan gradient clipping.
        for head in self.heads:
            norms = head.weight.norm(p=2, dim=1, keepdim=True)
            head.weight.mul_((MAX_NORM / norms.clamp_min(1e-12)).clamp(max=1.0))

## Cell 8 — Langkah multitask seimbang

Setiap optimizer step memilih uniform dari **semua empat task**.
Loader habis dibuat ulang agar reshuffle, tanpa `itertools.cycle`.
Konvensi epoch: `sum(ceil(n_train_task / batch_size))` update; panjang epoch ini
pilihan implementasi, tidak dirinci paper. Sumber kecil dapat diulang/disampel lebih sering.

In [ ]:
def train_epoch(model, optimizer, loaders, criteria, task_rng):
    model.train()
    iterators = {source: iter(loader) for source, loader in loaders.items()}
    updates = sum(len(loader) for loader in loaders.values())
    losses = 0.0
    task_steps = Counter()
    for _ in range(updates):
        source = task_rng.choice(SOURCES)
        try:
            batch = next(iterators[source])
        except StopIteration:
            iterators[source] = iter(loaders[source])
            batch = next(iterators[source])
        x, lengths, y = (tensor.to(DEVICE) for tensor in batch)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x, lengths, TASK_ID[source])
        loss = criteria[source](logits, y)
        if not torch.isfinite(loss).item():
            raise FloatingPointError(f"Loss tidak finite pada task {source}.")
        loss.backward()
        # max_norm=inf memeriksa norm gradient tanpa clipping.
        gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), float("inf"), error_if_nonfinite=True)
        optimizer.step()
        model.constrain_heads()
        losses += loss.item()
        task_steps[source] += 1
    return losses / updates, dict(task_steps)

## Cell 9 — Evaluasi epoch terakhir

Test fold hanya dievaluasi **setelah training selesai**; tidak dipakai memilih epoch,
hyperparameter, atau checkpoint. P/R/F1/support disimpan untuk kelas 0–4;
macro debt hanya kelas 1–4, termasuk kelas dengan F1 nol.

In [ ]:
@torch.no_grad()
def predict(model, loader, task_id):
    model.eval()
    predictions = []
    for x, lengths, _ in loader:
        logits = model(x.to(DEVICE), lengths.to(DEVICE), task_id)
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return np.asarray(predictions, dtype=np.int64)

def metric_rows(source, fold, y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(5)), zero_division=0,
    )
    debt = {f"debt_macro_{name}": float(values[list(DEBT_CLASSES)].mean())
            for name, values in (("precision", precision), ("recall", recall), ("f1", f1))}
    return [dict(source=source, fold=fold, class_id=i, label=LABELS[i],
                 precision=float(precision[i]), recall=float(recall[i]), f1=float(f1[i]),
                 support=int(support[i]), **debt) for i in range(5)]

## Cell 10 — Direktori hasil dan provenance

Setiap eksekusi membuat direktori UTC+UUID tersendiri di `Result/reproduce_baseline/`.
Metadata mencatat hash data, konfigurasi, package, perangkat, dan torch settings.
`config.json` bertanda started sampai seluruh fold terpilih selesai.

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:12]
RUN_DIR = ROOT / "Result" / "reproduce_baseline" / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)

def write_json(name, payload):
    path = RUN_DIR / name
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

# Simpan pengaturan yang dibutuhkan untuk menelusuri eksperimen.
config = {
    "status": "started",
    "cv_status": CV_STATUS,
    "run_folds": list(RUN_FOLDS),
    "n_splits": N_SPLITS,
    "seed": SEED,
    "fold_seed": "42 + fold (1-based)",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "max_len": MAX_LEN,
    "min_freq": MIN_FREQ,
    "embed_dim": EMBED_DIM,
    "embedding_init": "uniform(-0.25,0.25), PAD=0",
    "kernel_sizes": list(KERNEL_SIZES),
    "kernel_num": KERNEL_NUM,
    "dropout": DROPOUT,
    "optimizer": "Adadelta",
    "learning_rate": LEARNING_RATE,
    "rho": ADADELTA_RHO,
    "eps": ADADELTA_EPS,
    "max_head_row_norm": MAX_NORM,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "weight_formula_if_enabled": "n_train / (5 * n_train_class), no clipping",
    "save_checkpoints": SAVE_CHECKPOINTS,
    "checkpoint_selection": "final epoch per fold only",
    "task_id": TASK_ID,
    "labels": list(LABELS),
    "debt_classes": list(DEBT_CLASSES),
    "paper_doi": PAPER_DOI,
    "source_commit": SOURCE_COMMIT,
    "source_url": SOURCE_URL,
    "target": "Table4 scratch; unweighted default; not final fastText 0.611",
    "leakage_caveat": "row-level CV preserves duplicates and cross-source/project overlap",
    "preprocessing_scope": "detector-compatible; upstream bot/codeblock/non-English filtering not independently revalidated",
    "task_sampling": "uniform task each step; restart shuffled loaders",
    "epoch_updates": "sum(ceil(n_train_task/batch_size))",
    "std_ddof": 1,
    "overall_average": "equal domains per fold, then folds",
    "implementation_choices": [
        "fixed epochs without dev early stopping", "max_len", "min_freq", "seed",
        "embedding initialization", "Adadelta numerical defaults", "epoch length", "pooling mask",
    ],
}
write_json("config.json", config)
print("Hasil disimpan di:", RUN_DIR)

**Jejak data dan lingkungan.** Hash memudahkan pemeriksaan dataset; versi paket dan perangkat membantu mengulang eksperimen.

In [ ]:
data_hashes = {}
for source, url in FILES.items():
    data_hashes[source] = {
        "url": url,
        "sha256": hashlib.sha256(data_bytes[source]).hexdigest(),
        "rows": len(frames[source]),
    }
write_json("data_hashes.json", data_hashes)

packages = ("torch", "numpy", "pandas", "scikit-learn", "nltk")
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {name: importlib.metadata.version(name) for name in packages},
    "device": str(DEVICE),
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "torch_threads": torch.get_num_threads(),
    "torch_interop_threads": torch.get_num_interop_threads(),
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
    "deterministic_warn_only": torch.is_deterministic_algorithms_warn_only_enabled(),
    "cuda_matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
    "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
    "note": "warn_only=True; bitwise reproducibility lintas hardware/version tidak dijamin",
}
write_json("environment.json", environment)
fold_assignments.to_csv(RUN_DIR / "folds.csv", index=False)

## Cell 11 — Jalankan fold terpilih

Model, optimizer, vocabulary, dan loss baru setiap fold. Seed fold `42+fold`.
Tidak ada hasil training tersimpan di notebook ini. Cell ini menjalankan training nyata;
mode QUICK tetap menghasilkan **partial 10-fold CV**, bukan hasil lengkap paper.

In [ ]:
def prepare_fold(fold):
    fold_seed = SEED + fold
    set_seed(fold_seed)
    train_frames, test_frames = {}, {}
    for source in SOURCES:
        train_idx, test_idx = splits[source][fold - 1]
        train_frames[source] = frames[source].iloc[train_idx]
        test_frames[source] = frames[source].iloc[test_idx]

    vocab = build_vocab(train_frames)
    train_loaders = {
        source: make_loader(train_frames[source], vocab, True, fold_seed + TASK_ID[source])
        for source in SOURCES
    }
    test_loaders = {
        source: make_loader(test_frames[source], vocab, False, fold_seed)
        for source in SOURCES
    }
    criteria = {source: make_criterion(train_frames[source]["y"]) for source in SOURCES}
    return fold_seed, test_frames, vocab, train_loaders, test_loaders, criteria

**Latih satu fold.** Catat loss setiap epoch; data test belum digunakan.

In [ ]:
def fit_fold(model, optimizer, loaders, criteria, task_rng, fold, training_records):
    for epoch in range(1, EPOCHS + 1):
        loss, steps = train_epoch(model, optimizer, loaders, criteria, task_rng)
        training_records.append({
            "fold": fold,
            "epoch": epoch,
            "mean_loss": loss,
            "updates": sum(steps.values()),
            **{f"steps_{source}": steps.get(source, 0) for source in SOURCES},
        })
        pd.DataFrame(training_records).to_csv(RUN_DIR / "training_trace.csv", index=False)
        print(f"Fold {fold}/10 | epoch {epoch}/{EPOCHS} | loss={loss:.6f} | updates={steps}")

**Evaluasi satu fold.** Simpan prediksi setiap sumber setelah pelatihan selesai.

In [ ]:
def evaluate_fold(model, test_frames, test_loaders, fold, oof_parts, metric_records):
    for source in SOURCES:
        test = test_frames[source]
        y_pred = predict(model, test_loaders[source], TASK_ID[source])
        y_true = test["y"].to_numpy()
        assert len(y_pred) == len(test)
        oof_parts.append(pd.DataFrame({
            "source": source,
            "row_id": test["row_id"].to_numpy(),
            "ytrue": y_true,
            "ypred": y_pred,
            "fold": fold,
        }))
        metric_records.extend(metric_rows(source, fold, y_true, y_pred))

**Jalankan cross-validation.** Setiap fold memiliki model dan optimizer baru. Checkpoint disimpan hanya jika diaktifkan.

In [ ]:
oof_parts, metric_records, training_records = [], [], []
for fold in RUN_FOLDS:
    fold_seed, test_frames, vocab, train_loaders, test_loaders, criteria = prepare_fold(fold)
    task_rng = random.Random(fold_seed)
    model = MTTextCNN(len(vocab)).to(DEVICE)
    optimizer = torch.optim.Adadelta(
        model.parameters(), lr=LEARNING_RATE, rho=ADADELTA_RHO, eps=ADADELTA_EPS,
    )

    fit_fold(model, optimizer, train_loaders, criteria, task_rng, fold, training_records)
    evaluate_fold(model, test_frames, test_loaders, fold, oof_parts, metric_records)

    if SAVE_CHECKPOINTS:
        torch.save({
            "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},
            "vocab": vocab, "config": dict(config), "fold": fold, "epoch": EPOCHS,
        }, RUN_DIR / f"fold_{fold:02d}_final.pt")
    pd.concat(oof_parts, ignore_index=True).to_csv(RUN_DIR / "predictions.csv", index=False)
    pd.DataFrame(metric_records).to_csv(RUN_DIR / "fold_metrics.csv", index=False)
    del model, optimizer, train_loaders, test_loaders, criteria
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## Cell 12 — Cakupan dan ringkasan numerik

Mean/std dihitung dari skor tiap fold tanpa pembulatan antara. Std sampel (`ddof=1`);
satu fold memiliki std kosong/NaN, bukan estimasi variasi CV.
Overall merata-ratakan **empat domain dalam fold**, lalu merangkum antar-fold;
bukan berbobot jumlah sampel atau skor OOF gabungan. CSV tetap numerik presisi penuh.

In [ ]:
oof = pd.concat(oof_parts, ignore_index=True)
fold_metrics = pd.DataFrame(metric_records)
expected = fold_assignments[fold_assignments["fold"].isin(RUN_FOLDS)]
keys = ["source", "row_id", "fold"]
assert not oof.duplicated(["source", "row_id"]).any()
coverage = expected[keys].merge(oof[keys], on=keys, how="outer", indicator=True, validate="one_to_one")
assert coverage["_merge"].eq("both").all()
assert len(fold_metrics) == len(SOURCES) * len(RUN_FOLDS) * 5
for source in SOURCES:
    got = oof[oof["source"].eq(source)]
    assert set(got["fold"]) == set(RUN_FOLDS)
    if CV_STATUS == "full_10_fold":
        assert len(got) == len(frames[source])

per_class = fold_metrics.melt(
    id_vars=["source", "fold", "class_id"], value_vars=["precision", "recall", "f1"],
    var_name="metric", value_name="value",
)
macro = fold_metrics.drop_duplicates(["source", "fold"]).melt(
    id_vars=["source", "fold"], value_vars=["debt_macro_precision", "debt_macro_recall", "debt_macro_f1"],
    var_name="metric", value_name="value",
)
macro["class_id"] = -1  # -1 = agregasi kelas debt 1..4.
long_metrics = pd.concat([per_class, macro], ignore_index=True)
overall = long_metrics.groupby(["fold", "class_id", "metric"], as_index=False)["value"].mean()
overall["source"] = "overall_equal_domain"
summary = pd.concat([long_metrics, overall], ignore_index=True).groupby(
    ["source", "class_id", "metric"], as_index=False,
)["value"].agg(mean="mean", std="std", n_folds="count")
summary["cv_status"] = CV_STATUS
summary.to_csv(RUN_DIR / "summary.csv", index=False)
oof.to_csv(RUN_DIR / "predictions.csv", index=False)
fold_metrics.to_csv(RUN_DIR / "fold_metrics.csv", index=False)
config.update(status="completed", oof_rows=len(oof), completed_folds=list(RUN_FOLDS))
write_json("config.json", config)
print(f"{CV_STATUS}: cakupan {len(oof)}/{len(fold_assignments)} baris; fold {RUN_FOLDS}")
print(summary[summary["metric"].eq("debt_macro_f1")].to_string(index=False))

## Cell 13 — Referensi Table 4, bukan hasil eksekusi

Angka berikut **referensi paper**, bukan skor yang pernah dijalankan di notebook ini.
Perbandingan hanya deskriptif, tanpa tuning menggunakan test fold.
Jika loss berbobot/konfigurasi diubah atau CV parsial, tidak setara baseline lengkap.

Rujukan: [Li et al., DOI](https://doi.org/10.1007/s10664-023-10297-9), §4.1.1–4.1.2;
[Kim 2014](https://aclanthology.org/D14-1181/), §3.1.
Source detector resmi: commit `df412d88c12c42164dc7574389070f7ef2cc1d12`;
digunakan untuk preprocessing/urutan head, bukan menyamakan final tuned dengan scratch.

In [ ]:
table4_reference = pd.DataFrame({
    "source": ["code_comment", "commit_message", "pull_request", "issue", "overall_equal_domain"],
    "table4_scratch_debt_macro_f1_reference": [0.619, 0.475, 0.441, 0.453, 0.497],
})
comparison = summary[summary["metric"].eq("debt_macro_f1")][["source", "mean", "std", "n_folds", "cv_status"]].merge(
    table4_reference, on="source", validate="one_to_one",
)
comparison["difference_from_reference"] = comparison["mean"] - comparison["table4_scratch_debt_macro_f1_reference"]
comparison["use_class_weights"] = USE_CLASS_WEIGHTS
comparison.to_csv(RUN_DIR / "table4_reference_comparison.csv", index=False)
print(comparison.to_string(index=False))
print("Referensi terpisah; skor final FastText 0.611 bukan target scratch.")